# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Build a chat model client for any OpenAI-compatible endpoint.
2. Define a **tool** — a plain Python function — with the `@tool` decorator.
3. Create an agent with `create_agent` and run it.
4. Stream the agent's response token-by-token.

## Setup

Prerequisites: run `pip install -r requirements.txt` in the repository root, copy `.env.example` to `.env`, fill in `LLM_BASE_URL`, `LLM_API_KEY`, `LLM_MODEL`, and run `python scripts/check_endpoint.py`.

The cell below loads those variables from `.env` and builds the chat model client. `ChatOpenAI` speaks the OpenAI Chat Completions protocol, so the same code works with DeepSeek, OpenAI, a local Ollama server, or any other compatible endpoint — only the three environment variables change. `LLM_EXTRA_BODY` carries optional provider-specific request options (for DeepSeek it turns thinking mode off).

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave*. In LangChain these become the agent's **system prompt**.
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions. The function's **docstring becomes the tool description** the model reads when deciding whether to call it, and its type hints define the arguments.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [2]:
from langchain.tools import tool


@tool
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona", "Paris", "Berlin", "Tokyo", "Sydney",
        "New York City", "Cairo", "Cape Town", "Rio de Janeiro", "Bali",
    ]

Now we wire the model, the tool and the instructions together with `create_agent`. The agent runs a **tool-calling loop**: it sends the conversation to the model, executes any tool the model asks for, feeds the result back, and repeats until the model answers in plain text.

`agent.invoke` returns the full message history — user message, the model's tool call, the tool result, and the final reply. `reply_text` pulls the text out of the last message (some providers return content as a list of blocks rather than a single string).

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[get_destinations],
    system_prompt=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content


result = agent.invoke(
    {"messages": [{"role": "user", "content": "I'm looking for a warm beach destination. What do you recommend?"}]}
)
print(reply_text(result))

Based on your preference for a **warm beach destination**, here are my top recommendations from the available options:

1. **Bali** 🏝️ — The ultimate warm beach getaway. Expect tropical weather, stunning beaches (Kuta, Nusa Dua, Uluwatu), lush rice terraces, and a rich cultural scene. Perfect for both relaxation and adventure.

2. **Rio de Janeiro** 🌊 — Famous beaches like Copacabana and Ipanema, warm sunny weather year-round, vibrant culture, and incredible scenery (think Sugarloaf Mountain and Christ the Redeemer).

3. **Sydney** ☀️ — While it's on the temperate side, its famous beaches (Bondi, Manly) and warm summers make it a fantastic beach destination with plenty to do.

4. **Cape Town** 🌅 — Beautiful beaches like Camps Bay and Clifton, warm Mediterranean-style climate, and breathtaking scenery with Table Mountain as a backdrop.

5. **Barcelona** 🏖️ — Sunny Mediterranean beaches (Barceloneta) combined with amazing food, architecture, and nightlife.

**My top pick:** If you want a

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

`agent.astream(..., stream_mode="messages")` yields `(token, metadata)` pairs for every message chunk the graph produces. We print only the chunks that come from the model node (tool calls and tool results also flow through the stream). Jupyter supports top-level `await`, so the `async for` below runs as-is.

In [4]:
async for token, metadata in agent.astream(
    {"messages": [{"role": "user", "content": "Tell me about Tokyo as a travel destination"}]},
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
        print(token.content, end="", flush=True)
print()

Tokyo is a truly incredible travel destination! Here's what makes it so special:

## 🏙️ Overview
Tokyo is a dazzling blend of ultra-modern technology and deep-rooted tradition. It's one of the world's most exciting cities, where you can find serene ancient temples just blocks away from neon-lit shopping streets and futuristic skyscrapers.

## ✨ Highlights

**Iconic Neighborhoods**
- **Shibuya** – Famous for the Shibuya Crossing, one of the world's busiest pedestrian intersections, plus amazing shopping and nightlife.
- **Shinjuku** – Entertainment district with vibrant nightlife, the quirky Golden Gai bars, and lovely Shinjuku Gyoen Garden.
- **Asakusa** – Home to the historic Sensō-ji Temple, Tokyo's oldest temple, and traditional Nakamise shopping street.
- **Harajuku** – The heart of youth culture and eclectic fashion, right next to the peaceful Meiji Shrine.

**Culture & History**
- Ancient temples and shrines alongside imperial gardens
- World-class museums like the Tokyo National

## Summary

In this lesson you learned how to:

- **Create a model client** with `ChatOpenAI`, which talks to any OpenAI-compatible endpoint — the provider is just configuration.
- **Define a tool** with the `@tool` decorator, which turns a plain Python function (and its docstring) into something the model can call.
- **Create an agent** with `create_agent`, which wires the model, tools and instructions into a tool-calling loop.
- **Stream responses** with `astream` to print tokens as they arrive.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.